# Tool-Router Ladder: Stage D, escalation sweep (V1-19)

The best small student (1.5B, or 3B if Stage B shows it is better; `MODEL` below) drives tau2 episodes at catalog 80. Each turn it is scored from its own log-probs
(step confidence, src/logprobs.py); below the threshold, the teacher answers that turn instead
(src/router_node.py). Four thresholds: 0.5, 0.7, 0.85, 0.95.

**Settings:** GPU T4, Internet on. Secrets `HF_TOKEN` and `OPENROUTER_API_KEY`, both ticked.

**Run it in the background:** Save Version → **Save & Run All (Commit)**, as for Stage A.

**Quota:** escalated turns are teacher requests on the same ~1,000/day budget as the simulator, so
an episode costs more than in Stage A (~7–15 requests, more at higher thresholds). Each episode is
budgeted at its worst case (32 requests), so a day stops with some quota unused.
- **Never run this on the same UTC day as a local `harvest.py` run** (e.g. the teacher c40 baseline).
- Start after 7 PM CDT, when the quota resets.

**If it prints `STOPPED EARLY`:** the provider or the network is down. Stop for the day.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "OPENROUTER_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "OPENROUTER_API_KEY"])

In [ ]:
# Pinned: every forced-eval row so far is vLLM 0.30.0 (with a CUDA 13 torch).
# Kaggle's image now ships a CUDA 12.8 torchaudio, and transformers imports torchaudio
# whenever it is installed, so it crashes against that torch. Nothing here uses audio.
!pip install -q vllm==0.30.0
!pip uninstall -y -q torchaudio

# Step out of the working dir before deleting anything under it, so a re-run in
# the same session does not leave the kernel in a dead directory.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/tau2-bench", ignore_errors=True)
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/sierra-research/tau2-bench.git /kaggle/working/tau2-bench
!cd /kaggle/working/tau2-bench && git checkout -q 672227c6b6676edc20d57ea53b7000262aae77b9 && pip install -q -e .

!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
HF_USER = "madhusiddharths1"   # your Hugging Face username
SYNC = f"{HF_USER}/tool-router-results"
# Stage D escalates from the best small model at catalog 80 (plan §5). 1.5B unless Stage B's
# 3B c80 result beats it; change this BEFORE the first daily run, never midway through.
MODEL = "qwen15b"


### Smoke run

2 tasks, 1 seed, a 2-turn simulator, at catalog 16 (smoke mode allows only 16) and threshold 0.95 so the escalation path is exercised; about
10 API requests. It checks that the vLLM server returns log-probs by token id, that turns are
scored and routed, and that an escalated turn reaches the teacher and gets scored.

In [ ]:
import json, pathlib, subprocess

# subprocess rather than `!`, so a failed smoke raises and stops a Save & Run All.
smoke = subprocess.run(
    ["python", "src/eval_free.py", "--config", f"configs/{MODEL}.yaml", "--catalog", "16",
     "--adapter", f"{HF_USER}/tool-router-{MODEL}-sft", "--smoke",
     "--backend", "server", "--concurrency", "2", "--escalate", "0.95"]
)
assert smoke.returncode == 0, "smoke run failed: fix it before the daily run spends quota"
rows = [json.loads(l) for f in pathlib.Path("results").glob("free-*-esc0.95-*.smoke.jsonl")
        for l in f.read_text().splitlines() if l.strip()]
print("smoke episodes:", [(r["task_id"], r["reward"], r["student_turns"], r["escalated_turns"]) for r in rows])
assert rows and sum(r["escalated_turns"] for r in rows) > 0, "no turn escalated at 0.95: check log-probs"

### Daily run: the four thresholds, in order

Each command finishes its threshold or stops at the day's budget; later commands then stop
immediately. Re-run this notebook each day; finished thresholds are skipped.

**Order:** 0.7, 0.95, 0.5, 0.85, so that if quota runs short the finished points are spread across
the curve. Its two ends already exist: threshold 0 is Stage A 1.5B c80 (2.7%) and "always
escalate" is the teacher c80 baseline (54.7%).

**Timing:** note the minutes per threshold printed at the end and send them back.

In [ ]:
import time
started = time.time()
for t in ["0.7", "0.95", "0.5", "0.85"]:
    print(f"===== {MODEL} catalog 80 threshold {t} =====")
    t0 = time.time()
    !python src/eval_free.py --config configs/{MODEL}.yaml --catalog 80 --adapter "$HF_USER/tool-router-{MODEL}-sft" --hub-sync "$SYNC" --backend server --concurrency 6 --escalate {t}
    print(f"[timing] threshold {t}: {(time.time() - t0) / 60:.1f} min")
print(f"[timing] session total: {(time.time() - started) / 3600:.2f} h")